# Module 8.1 — Working with Databases

### Python Mastery · Track 8: Real-World Python

Most real applications store data in a database. This module covers `sqlite3` from the standard library, the DB-API 2.0 conventions that all Python database drivers share, safe parameterised queries, transactions, and an orientation to SQLAlchemy's Core and ORM layers.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it (`Shift + Enter`).
- Everything here runs fully: `sqlite3` is in the standard library, and the examples use an in-memory or temporary database so nothing is left behind. SQLAlchemy is also demonstrated with real, runnable code.
- Attempt the exercises before consulting the solutions at the end.

### Learning objectives

After completing this module you will be able to:

1. Connect to SQLite and execute statements through a cursor.
2. Use parameterised queries to prevent SQL injection.
3. Manage transactions with commit and rollback.
4. Read results with fetch methods and row factories.
5. Describe SQLAlchemy Core and ORM and write a simple ORM model.

**Prerequisites:** Tracks 1 to 6.

---

## Part 1 · Connecting and the DB-API 2.0

Python database drivers follow a shared standard, **DB-API 2.0**, so the patterns you learn with `sqlite3` carry over to PostgreSQL, MySQL, and others. The core objects are a **connection** (a session with the database) and a **cursor** (used to execute statements and fetch results). SQLite can use a file on disk or an in-memory database (`":memory:"`), which is ideal for experiments and tests.

In [ ]:
import sqlite3

# An in-memory database: fast, isolated, and discarded when the connection closes.
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Execute a statement to create a table.
cursor.execute("""
    CREATE TABLE books (
        id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        author TEXT NOT NULL,
        year INTEGER
    )
""")

print("table created")
print("sqlite version:", sqlite3.sqlite_version)
print("connection:", type(conn).__name__, "| cursor:", type(cursor).__name__)

## Part 2 · Parameterised Queries and SQL Injection Safety

The single most important database rule: **never build SQL by string formatting with user input.** Doing so allows **SQL injection**, where crafted input changes the meaning of your query. Instead, pass values as **parameters** using `?` placeholders (SQLite's style); the driver sends the values separately, so they can never be interpreted as SQL. This is both safer and often faster.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, author TEXT, year INTEGER)")

# SAFE: values passed as parameters via ? placeholders.
cursor.execute(
    "INSERT INTO books (title, author, year) VALUES (?, ?, ?)",
    ("Dune", "Herbert", 1965),
)

# executemany inserts many rows efficiently, each as a parameter tuple.
more_books = [
    ("Neuromancer", "Gibson", 1984),
    ("Snow Crash", "Stephenson", 1992),
    ("The Left Hand of Darkness", "Le Guin", 1969),
]
cursor.executemany("INSERT INTO books (title, author, year) VALUES (?, ?, ?)", more_books)
conn.commit()

print("rows inserted:", cursor.rowcount, "(from the last executemany)")

# A query with a parameter: the value is data, never code.
search_year = 1980
cursor.execute("SELECT title, author FROM books WHERE year > ?", (search_year,))
print(f"books after {search_year}:", cursor.fetchall())

In [ ]:
# Why this matters: a hostile "title" cannot break out of the parameter.
hostile = "x'); DROP TABLE books; --"     # a classic injection attempt

# Passed as a parameter, it is stored as a literal string, doing no harm.
cursor.execute("INSERT INTO books (title, author, year) VALUES (?, ?, ?)",
               (hostile, "nobody", 2000))
conn.commit()

cursor.execute("SELECT COUNT(*) FROM books")
print("table still exists; row count:", cursor.fetchone()[0])
print("the injection attempt was stored as plain text, not executed")

## Part 3 · Reading Results and Row Factories

A cursor offers three ways to retrieve rows: `fetchone()` for a single row, `fetchmany(n)` for a batch, and `fetchall()` for everything. By default each row is a tuple, so you access columns by position. Setting `conn.row_factory = sqlite3.Row` lets you access columns by **name** instead, which is far more readable. You can also iterate a cursor directly, row by row, which is memory-friendly for large results.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row          # rows accessible by column name
cursor = conn.cursor()
cursor.execute("CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, author TEXT, year INTEGER)")
cursor.executemany(
    "INSERT INTO books (title, author, year) VALUES (?, ?, ?)",
    [("Dune", "Herbert", 1965), ("Neuromancer", "Gibson", 1984), ("Snow Crash", "Stephenson", 1992)],
)
conn.commit()

# fetchone returns the next row (or None when exhausted).
cursor.execute("SELECT * FROM books ORDER BY year")
first = cursor.fetchone()
print("oldest book by name:", first["title"], "by", first["author"], f"({first['year']})")

# Iterate the cursor directly: one row at a time, easy on memory.
cursor.execute("SELECT title, year FROM books ORDER BY year")
print("all books:")
for row in cursor:
    print(f"  {row['year']}: {row['title']}")

## Part 4 · Transactions: Commit and Rollback

A **transaction** groups statements so they succeed or fail together. Changes are not permanent until you `commit()`; if something goes wrong, `rollback()` undoes everything since the last commit, leaving the database consistent. Using the connection as a context manager commits automatically on success and rolls back on an exception, which is the cleanest pattern.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT, balance INTEGER)")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("Asha", 100), ("Ben", 50)])
conn.commit()

def transfer(conn, sender, recipient, amount):
    """Move money atomically: both updates happen, or neither does."""
    try:
        with conn:                       # commits on success, rolls back on error
            conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, sender))
            balance = conn.execute("SELECT balance FROM accounts WHERE name = ?", (sender,)).fetchone()[0]
            if balance < 0:
                raise ValueError("insufficient funds")     # triggers rollback
            conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, recipient))
        return "transfer succeeded"
    except ValueError as e:
        return f"transfer failed and rolled back: {e}"

print(transfer(conn, "Asha", "Ben", 30))     # succeeds
print(transfer(conn, "Asha", "Ben", 1000))   # fails, rolls back

for row in conn.execute("SELECT name, balance FROM accounts"):
    print(f"  {row[0]}: {row[1]}")
print("balances remain consistent because the failed transfer was undone entirely")

## Part 5 · SQLAlchemy: Core and ORM

For larger applications, the **SQLAlchemy** library offers two layers above raw drivers. **Core** provides a Pythonic way to build SQL expressions while staying close to tables and queries. The **ORM** (Object-Relational Mapper) goes further, mapping database rows to Python objects, so you work with classes and attributes instead of SQL. The ORM is more convenient for application data models; Core is closer to the SQL and useful for complex queries or reporting.

The cell below uses the ORM to define a model, create the table, add rows, and query, all against an in-memory SQLite database.

In [ ]:
from sqlalchemy import create_engine, String, Integer
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

# The base class for ORM models.
class Base(DeclarativeBase):
    pass

# A model: each instance corresponds to a row in the 'books' table.
class Book(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    title: Mapped[str] = mapped_column(String)
    author: Mapped[str] = mapped_column(String)
    year: Mapped[int] = mapped_column(Integer)

    def __repr__(self):
        return f"Book({self.title!r} by {self.author}, {self.year})"

# An engine manages connections; here, in-memory SQLite.
engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)        # create tables from the models

# A session is the ORM's unit of work.
with Session(engine) as session:
    session.add_all([
        Book(title="Dune", author="Herbert", year=1965),
        Book(title="Neuromancer", author="Gibson", year=1984),
        Book(title="Snow Crash", author="Stephenson", year=1992),
    ])
    session.commit()

    # Query with Python objects, not raw SQL.
    from sqlalchemy import select
    recent = session.scalars(select(Book).where(Book.year > 1980).order_by(Book.year)).all()
    print("books after 1980:")
    for book in recent:
        print("  ", book)

In [ ]:
from sqlalchemy import create_engine, text

# SQLAlchemy Core: build and run SQL expressions while staying close to the database.
engine = create_engine("sqlite:///:memory:")
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE items (id INTEGER PRIMARY KEY, name TEXT, price REAL)"))
    conn.execute(
        text("INSERT INTO items (name, price) VALUES (:name, :price)"),
        [{"name": "pen", "price": 1.5}, {"name": "notebook", "price": 4.0}],
    )
    conn.commit()

    # Named parameters (:name style) keep queries safe, as with sqlite3's ? placeholders.
    result = conn.execute(text("SELECT name, price FROM items WHERE price < :limit"), {"limit": 3.0})
    print("cheap items:", [tuple(row) for row in result])

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Create, insert, and read (Easy)

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE colors (name TEXT, hex TEXT)")
conn.execute("INSERT INTO colors VALUES (?, ?)", ("teal", "#008080"))
conn.commit()
print(conn.execute("SELECT * FROM colors").fetchall())

### Example 2 — Parameterised search (Easy)

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE nums (n INTEGER)")
conn.executemany("INSERT INTO nums VALUES (?)", [(i,) for i in range(10)])
conn.commit()
threshold = 5
rows = conn.execute("SELECT n FROM nums WHERE n >= ?", (threshold,)).fetchall()
print("values >=", threshold, "->", [r[0] for r in rows])

### Example 3 — Row factory for named access (Medium)

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("CREATE TABLE people (name TEXT, age INTEGER)")
conn.executemany("INSERT INTO people VALUES (?, ?)", [("Asha", 30), ("Ben", 25)])
conn.commit()
for row in conn.execute("SELECT * FROM people ORDER BY age"):
    print(f"{row['name']} is {row['age']}")

### Example 4 — A transaction that rolls back (Medium)

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE stock (item TEXT, qty INTEGER)")
conn.execute("INSERT INTO stock VALUES (?, ?)", ("widgets", 5))
conn.commit()

try:
    with conn:
        conn.execute("UPDATE stock SET qty = qty - ? WHERE item = ?", (10, "widgets"))
        qty = conn.execute("SELECT qty FROM stock WHERE item = ?", ("widgets",)).fetchone()[0]
        if qty < 0:
            raise ValueError("cannot go negative")
except ValueError as e:
    print("rolled back:", e)

print("qty after rollback:", conn.execute("SELECT qty FROM stock").fetchone()[0])

### Example 5 — Aggregation and grouping (Difficult)

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE sales (region TEXT, amount INTEGER)")
conn.executemany("INSERT INTO sales VALUES (?, ?)", [
    ("North", 100), ("South", 200), ("North", 150), ("South", 50), ("East", 300),
])
conn.commit()

# GROUP BY with an aggregate, ordered by the total.
rows = conn.execute("""
    SELECT region, SUM(amount) AS total, COUNT(*) AS deals
    FROM sales
    GROUP BY region
    ORDER BY total DESC
""").fetchall()
for region, total, deals in rows:
    print(f"{region}: total {total} over {deals} deals")

### Example 6 — An ORM relationship-free CRUD cycle (Difficult)

In [ ]:
from sqlalchemy import create_engine, String, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

class Base(DeclarativeBase):
    pass

class Task(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    title: Mapped[str] = mapped_column(String)
    done: Mapped[int] = mapped_column(Integer, default=0)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)

with Session(engine) as session:
    # Create
    t = Task(title="write notebook")
    session.add(t)
    session.commit()

    # Read
    fetched = session.scalars(select(Task).where(Task.title == "write notebook")).one()
    print("created task id:", fetched.id, "done:", fetched.done)

    # Update
    fetched.done = 1
    session.commit()

    # Confirm update, then Delete
    print("after update, done:", session.get(Task, fetched.id).done)
    session.delete(fetched)
    session.commit()
    print("remaining tasks:", session.scalars(select(Task)).all())

---

## Exercises

**Exercise 1 (Easy).** Create an in-memory SQLite table `fruits(name TEXT, count INTEGER)`, insert two rows, and print all rows.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Using a parameterised query, insert a row supplied as variables, then select it back by name with another parameterised query.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Set `conn.row_factory = sqlite3.Row`, create a `movies(title, year)` table with three rows, and print each as `"title (year)"` ordered by year.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a function that, inside a `with conn:` block, inserts two related rows; raise an error after the first insert to demonstrate that neither row remains (rollback).

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Using SQLAlchemy ORM, define a `User` model with `id` and `name`, create the table in an in-memory database, add two users, and query for the user whose name equals `"Asha"`.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE fruits (name TEXT, count INTEGER)")
conn.executemany("INSERT INTO fruits VALUES (?, ?)", [("apple", 3), ("pear", 5)])
conn.commit()
print(conn.execute("SELECT * FROM fruits").fetchall())

**Solution 2**

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (name TEXT, email TEXT)")
name, email = "Asha", "asha@example.com"
conn.execute("INSERT INTO users VALUES (?, ?)", (name, email))
conn.commit()
print(conn.execute("SELECT * FROM users WHERE name = ?", (name,)).fetchall())

**Solution 3**

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("CREATE TABLE movies (title TEXT, year INTEGER)")
conn.executemany("INSERT INTO movies VALUES (?, ?)",
                 [("Arrival", 2016), ("Dune", 2021), ("Blade Runner", 1982)])
conn.commit()
for row in conn.execute("SELECT * FROM movies ORDER BY year"):
    print(f"{row['title']} ({row['year']})")

**Solution 4**

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE log (msg TEXT)")
conn.commit()

def write_two(conn):
    try:
        with conn:
            conn.execute("INSERT INTO log VALUES (?)", ("first",))
            raise RuntimeError("boom before second insert")
    except RuntimeError as e:
        print("rolled back:", e)

write_two(conn)
print("rows remaining:", conn.execute("SELECT COUNT(*) FROM log").fetchone()[0])

**Solution 5**

In [ ]:
from sqlalchemy import create_engine, String, Integer, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String)

engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(engine)

with Session(engine) as session:
    session.add_all([User(name="Asha"), User(name="Ben")])
    session.commit()
    found = session.scalars(select(User).where(User.name == "Asha")).one()
    print("found:", found.id, found.name)

---

## Summary and Key Points

- Python database access follows DB-API 2.0: a connection represents a session and a cursor executes statements and fetches results; SQLite supports file-based and in-memory (`":memory:"`) databases.
- Always use parameterised queries with placeholders (`?` in SQLite) and pass values separately; never format user input into SQL, which invites injection. `executemany` inserts many rows efficiently.
- Retrieve rows with `fetchone`, `fetchmany`, or `fetchall`, or iterate the cursor; `sqlite3.Row` enables readable access by column name.
- Transactions group statements: changes become permanent on `commit()` and are undone by `rollback()`; using the connection as a context manager commits on success and rolls back on error.
- SQLAlchemy adds two layers: Core (Pythonic SQL expressions) and the ORM (maps rows to objects), with the ORM convenient for application models and Core useful for complex queries.

### Next module: 8.2 — HTTP, REST & Consuming APIs

The next module covers talking to web services: HTTP fundamentals, making requests with `requests` and `httpx`, working with JSON APIs, and handling pagination, retries, and timeouts.